# Sionna RT — USD / Omniverse Scene Builder
## Build a radio scene directly from a USD (NVIDIA Omniverse) stage — no OpenStreetMap

This is the USD/Omniverse counterpart to `sionna019_scene_builder_london.ipynb`.
Instead of downloading OSM building/road/vegetation footprints and extruding them,
it reads real mesh geometry + materials directly out of a `.usd`/`.usda`/`.usdc`
stage (e.g. exported from Omniverse, or imported there from another tool) and
converts it into the same `mat_plys` → `scene.xml` pipeline the OSM builder uses,
so the output is a drop-in for `sionna2_915mhz_dem_simulation_london.ipynb` /
`sionna018_neural_calibration_london.ipynb`.

**Status: untested against a real USD file.** `DEMO_MODE=True` (CELL 0) generates
a small synthetic USD-like scene in memory so every cell is runnable today without
`usd-core` installed and without an Omniverse export to test against. Once you
have a real `.usd` file, set `DEMO_MODE=False` and `USD_FILE` to its path.

---

## Build Sequence
1. CELL 0 — Configuration (paths, `PROJECTION_CRS`, `DEMO_MODE`)
2. CELL 1 — Imports (`pxr`/usd-core, optional `trimesh`)
3. CELL 2 — Load USD stage → flatten mesh prims to world-space verts/faces
4. CELL 3 — Map USD material/shader names → ITU-R P.2040-2 material names
5. CELL 4 — Write per-mesh PLYs (`meshes/`)
6. CELL 5 — Write `scene.xml` (Sionna 0.19 Mitsuba format — same writer as the OSM builder)
7. CELL 6 — 2D top-down preview of extracted geometry
8. CELL 7 — Load into Sionna RT and sanity-check materials/bbox

— Scene is then ready for `sionna2_915mhz_dem_simulation_london.ipynb` /
`sionna018_neural_calibration_london.ipynb` (point `SCENE_XML` at the output).


## CELL 0 — Configuration

In [ ]:
# ============================================================
# CELL 0 — CONFIG  (edit this block only)
# ============================================================
import os

SCENARIO_NAME = 'london_omniverse_usd'
CITY_NAME     = 'London'

# ── Demo mode ─────────────────────────────────────────────────────────────────
# True  -> build a small synthetic in-memory scene (no usd-core, no real file
#          needed) so the rest of the notebook is runnable today.
# False -> load USD_FILE for real via pxr/usd-core.
DEMO_MODE = True
USD_FILE  = '/path/to/your/omniverse_scene.usd'   # only used when DEMO_MODE=False

BASE_DIR  = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', SCENARIO_NAME))
SCENE_DIR = os.path.join(BASE_DIR, 'scene_usd')
MESH_DIR  = os.path.join(SCENE_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)

# ── Coordinate system ─────────────────────────────────────────────────────────
# USD scenes are usually authored in a local Cartesian frame (metres) already,
# with no inherent GPS anchor -- unlike OSM, which is always WGS84-derived.
# If your USD stage carries real-world georeference metadata, set
# USD_HAS_GEOREFERENCE=True and fill in the anchor; otherwise the scene origin
# is just wherever the USD stage's own (0,0,0) is, and you must align it to
# your TX/RX GPS positions manually (e.g. by placing TX at a known USD prim).
USD_HAS_GEOREFERENCE = False
ANCHOR_LON = -0.13399    # only used if USD_HAS_GEOREFERENCE=True
ANCHOR_LAT =  51.5305

# Must match PROJECTION_CRS in sionna2_915mhz_dem_simulation_london.ipynb /
# sionna019_scene_builder_london.ipynb / sionna018_neural_calibration_london.ipynb
PROJECTION_CRS  = 'bng'    # 'bng' | 'utm30n'
_PROJECTION_EPSG_MAP = {'bng': 27700, 'utm30n': 32630}
UTM_EPSG = _PROJECTION_EPSG_MAP.get(PROJECTION_CRS, 27700)

FREQUENCY_HZ = 915.95e6

print(f'Scenario   : {SCENARIO_NAME}')
print(f'Demo mode  : {DEMO_MODE}')
print(f'USD file   : {USD_FILE if not DEMO_MODE else "(none -- synthetic demo scene)"}')
print(f'Scene dir  : {SCENE_DIR}')
print(f'Projection : {PROJECTION_CRS}  ->  EPSG:{UTM_EPSG}')


## CELL 1 — Imports

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, json
import numpy as np

try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
print(f'trimesh  : {"OK" if _HAS_TRIMESH else "not available -- falls back to ASCII PLY writer"}')

_HAS_USD = False
if not DEMO_MODE:
    try:
        from pxr import Usd, UsdGeom, UsdShade, Gf
        _HAS_USD = True
        print('pxr (usd-core) : OK')
    except ImportError as _e:
        print(f'pxr (usd-core) : NOT AVAILABLE -- {_e}')
        print('  Install with:  pip install usd-core')
        print('  Falling back to DEMO_MODE behaviour for this run.')
else:
    print('pxr (usd-core) : skipped (DEMO_MODE=True)')


## CELL 2 — Load USD Stage → Flatten Mesh Prims
Real path: opens the stage with `Usd.Stage.Open`, walks every `UsdGeom.Mesh` prim,
applies its computed world transform (`UsdGeom.XformCache`), and triangulates
`faceVertexCounts`/`faceVertexIndices` (USD meshes can have n-gon faces, not just
triangles) into `(verts, tri_faces, material_name)` per prim.

Demo path: builds 4 synthetic box prims (3 "buildings" + 1 "ground slab") with
material names chosen to exercise the ITU mapping in CELL 3, standing in for
what `UsdShadeMaterialBindingAPI` would have returned from a real stage.

In [ ]:
# ============================================================
# CELL 2 — LOAD USD STAGE → WORLD-SPACE MESH PRIMS
# ============================================================
# Output: usd_meshes = [{'name': str, 'verts': (N,3) float32,
#                         'faces': (M,3) int32, 'material': str}, ...]

def _box_mesh(cx, cy, cz, sx, sy, sz):
    """Axis-aligned box centred at (cx,cy,cz), half-extents (sx,sy,sz)/2."""
    x0, x1 = cx - sx/2, cx + sx/2
    y0, y1 = cy - sy/2, cy + sy/2
    z0, z1 = cz, cz + sz
    v = np.array([
        [x0,y0,z0],[x1,y0,z0],[x1,y1,z0],[x0,y1,z0],  # bottom
        [x0,y0,z1],[x1,y0,z1],[x1,y1,z1],[x0,y1,z1],  # top
    ], dtype=np.float32)
    f = np.array([
        [0,1,2],[0,2,3],          # bottom
        [4,6,5],[4,7,6],          # top
        [0,4,5],[0,5,1],          # sides
        [1,5,6],[1,6,2],
        [2,6,7],[2,7,3],
        [3,7,4],[3,4,0],
    ], dtype=np.int32)
    return v, f

usd_meshes = []

if DEMO_MODE or not _HAS_USD:
    print('Building synthetic demo scene (3 buildings + ground slab) ...')
    v, f = _box_mesh(0, 0, 0, 400, 300, 0.2);      usd_meshes.append({'name': 'ground',     'verts': v, 'faces': f, 'material': 'Concrete_Ground'})
    v, f = _box_mesh(-80, 60, 0, 40, 30, 22.0);    usd_meshes.append({'name': 'building_A', 'verts': v, 'faces': f, 'material': 'Brick_Facade'})
    v, f = _box_mesh(60, -40, 0, 50, 50, 35.0);    usd_meshes.append({'name': 'building_B', 'verts': v, 'faces': f, 'material': 'Glass_Curtain_Wall'})
    v, f = _box_mesh(20, 90, 0, 30, 20, 12.0);     usd_meshes.append({'name': 'building_C', 'verts': v, 'faces': f, 'material': 'Metal_Cladding'})
    print(f'  {len(usd_meshes)} synthetic prims created (demo data, NOT a real scene)')
else:
    print(f'Opening USD stage: {USD_FILE}')
    stage = Usd.Stage.Open(USD_FILE)
    assert stage is not None, f'Could not open USD stage: {USD_FILE}'
    xform_cache = UsdGeom.XformCache()

    def _triangulate(face_counts, face_indices):
        tris = []
        idx = 0
        for n in face_counts:
            poly = face_indices[idx: idx + n]
            for k in range(1, n - 1):
                tris.append([poly[0], poly[k], poly[k + 1]])
            idx += n
        return np.asarray(tris, dtype=np.int32)

    def _bound_material_name(prim):
        try:
            rel = UsdShade.MaterialBindingAPI(prim).ComputeBoundMaterial()
            mat = rel[0] if isinstance(rel, tuple) else rel
            if mat and mat.GetPrim().IsValid():
                return mat.GetPrim().GetName()
        except Exception:
            pass
        return 'DEFAULT'

    for prim in stage.Traverse():
        if not prim.IsA(UsdGeom.Mesh):
            continue
        mesh = UsdGeom.Mesh(prim)
        pts = np.asarray(mesh.GetPointsAttr().Get(), dtype=np.float64)
        if pts is None or len(pts) == 0:
            continue
        counts = mesh.GetFaceVertexCountsAttr().Get()
        idxs   = mesh.GetFaceVertexIndicesAttr().Get()
        faces  = _triangulate(counts, idxs)

        world = xform_cache.GetLocalToWorldTransform(prim)
        world_np = np.array(world).reshape(4, 4)
        pts_h = np.hstack([pts, np.ones((len(pts), 1))])
        pts_world = (pts_h @ world_np)[:, :3].astype(np.float32)

        usd_meshes.append({
            'name': prim.GetName(),
            'verts': pts_world,
            'faces': faces,
            'material': _bound_material_name(prim),
        })
    print(f'  {len(usd_meshes)} mesh prims extracted from stage')

for m in usd_meshes:
    print(f"  {m['name']:<16} verts={len(m['verts']):>5}  faces={len(m['faces']):>5}  material={m['material']}")


## CELL 3 — Map USD Materials → ITU-R P.2040-2 Material Names
USD/Omniverse materials carry visual (PBR/MaterialX) names, not RF electromagnetic
properties — there is no built-in "this is concrete for radio purposes" semantic.
This cell matches each USD material's **name** against the same keyword set the
OSM pipeline uses, falling back to a default if nothing matches. If your USD
materials use opaque names (e.g. `Material_007`), you'll need to either rename
them before export or extend `_USD_MAT_KEYWORDS` with project-specific aliases —
matching by `diffuseColor`/MaterialX graph instead of name is a possible future
upgrade, not implemented here.

In [ ]:
# ============================================================
# CELL 3 — USD MATERIAL NAME → ITU-R MATERIAL MAPPING
# ============================================================
_USD_MAT_KEYWORDS = {
    'itu_concrete'  : ('concrete', 'cement', 'paving'),
    'itu_brick'     : ('brick',),
    'itu_glass'     : ('glass', 'window', 'curtain_wall', 'curtainwall'),
    'itu_metal'     : ('metal', 'steel', 'aluminium', 'aluminum', 'cladding'),
    'itu_wood'      : ('wood', 'timber', 'plywood'),
    'itu_plasterboard': ('plaster', 'drywall', 'gypsum'),
    'itu_marble'    : ('marble',),
    'itu_asphalt'   : ('asphalt', 'tarmac', 'road'),
    'itu_vegetation': ('vegetation', 'tree', 'foliage', 'grass', 'leaf'),
    'itu_water'     : ('water', 'pond', 'river', 'lake'),
    'itu_wet_ground': ('ground', 'soil', 'dirt', 'terrain'),
}
_DEFAULT_ITU_MAT = 'itu_concrete'

def _match_usd_material(usd_mat_name):
    n = usd_mat_name.lower()
    for itu_name, kws in _USD_MAT_KEYWORDS.items():
        if any(kw in n for kw in kws):
            return itu_name
    return None

print('USD material -> ITU mapping:')
mesh_itu_material = {}
for m in usd_meshes:
    matched = _match_usd_material(m['material'])
    itu_name = matched or _DEFAULT_ITU_MAT
    mesh_itu_material[m['name']] = itu_name
    flag = '' if matched else '  (no keyword match -- using default)'
    print(f"  {m['material']:<24} -> {itu_name:<16}{flag}")


## CELL 4 — Write Per-Mesh PLYs
Same writer (`_write_ply`) the OSM builder's CELL 4 uses, so downstream tooling
(CELL 5's `scene.xml` writer, `blender_to_sionna2_converter.py`, the simulation
notebooks) sees identical PLY files regardless of where the geometry came from.

In [ ]:
# ============================================================
# CELL 4 — WRITE PLYs
# ============================================================
def _write_ply(verts, faces, path):
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(path)
        return
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:  f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for fc in faces: f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')

# mat_plys: {itu_material_name: [(relative_ply_path, role), ...]} -- same shape
# the OSM builder's CELL 5 scene.xml writer expects.
mat_plys = {}
ground_ply = None

for m in usd_meshes:
    itu_name = mesh_itu_material[m['name']]
    fname = f"usd_{m['name']}.ply"
    fpath = os.path.join(MESH_DIR, fname)
    _write_ply(m['verts'], m['faces'], fpath)
    rel_path = os.path.basename(MESH_DIR) + '/' + fname

    is_ground = itu_name in ('itu_wet_ground', 'itu_asphalt') and m['name'] == 'ground'
    if is_ground:
        ground_ply = rel_path
        continue
    mat_plys.setdefault(itu_name, []).append((rel_path, 'usd_import'))

print(f'Wrote {len(usd_meshes)} PLY(s) -> {MESH_DIR}')
print(f'Ground PLY : {ground_ply or "(none found -- CELL 5 will fall back to a flat terrain.ply)"}')
print(f'mat_plys   : {{ {", ".join(f"{k}: {len(v)}" for k, v in mat_plys.items())} }}')

# CELL 5 (ported from the OSM builder) writes a fixed "terrain.ply" shape --
# if the USD scene had its own ground mesh, reuse it under that name so the
# scene doesn't end up with two overlapping ground planes.
import shutil
terrain_ply_path = os.path.join(MESH_DIR, 'terrain.ply')
if ground_ply is not None:
    shutil.copyfile(os.path.join(BASE_DIR, ground_ply), terrain_ply_path)
    print(f'Copied USD ground mesh -> {terrain_ply_path}')
elif not os.path.exists(terrain_ply_path):
    _gv, _gf = _box_mesh(0, 0, 0, 1000, 1000, 0.0)
    _write_ply(_gv, _gf, terrain_ply_path)
    print(f'No ground mesh in USD scene -- wrote flat 1km x 1km placeholder terrain.ply')


## CELL 5 — Write scene.xml (Sionna 0.19 Mitsuba format)
Identical material-parameter table, colour table, and `_guard_bsdf_refs` safety
net as `sionna019_scene_builder_london.ipynb` CELL 5 — only the source of
`mat_plys` differs (USD meshes here, OSM footprints there), so the XML output
format is a guaranteed drop-in.

In [ ]:
# ============================================================
# CELL 5 — WRITE SCENE.XML  (Mitsuba 2.1.0 / Sionna 0.19)
# ============================================================
_ITU_P2040_PARAMS = {
    #                    a       b       c        d      s     xpd
    'itu_concrete' : (5.31,  0.000,  0.0326, 0.8095, 0.30, 0.10),
    'itu_brick'    : (3.91,  0.000,  0.0238, 0.0000, 0.25, 0.10),
    'itu_glass'    : (6.27,  0.000,  0.0043, 1.1925, 0.10, 0.05),
    'itu_plywood'  : (1.99,  0.000,  0.0047, 1.0718, 0.20, 0.10),
    'itu_metal'    : (1.00,  0.000,  1.0e7,  0.0000, 0.05, 0.05),
    'itu_wet_ground'       : (30.0, -0.400, 0.1500, 1.3000, 0.35, 0.05),
    'itu_water'            : (80.0,  0.000, 0.0100, 0.0000, 0.02, 0.05),
    'itu_medium_dry_ground': (15.0, -0.100, 0.0350, 1.6300, 0.10, 0.05),
    'itu_very_dry_ground'  : ( 3.0,  0.000, 0.00015,2.5200, 0.10, 0.05),
    'itu_vegetation'       : ( 1.50, 0.000, 0.0020, 0.5000, 0.40, 0.50),
    'itu_asphalt'          : ( 2.56, 0.000, 0.0050, 0.0000, 0.30, 0.15),
}
_f_ghz_c5 = float(globals().get('FREQUENCY_HZ', 915.95e6)) / 1e9
ITU_MATERIALS = {}
for _mn, (_a, _b, _c, _d, _s, _xpd) in _ITU_P2040_PARAMS.items():
    ITU_MATERIALS[_mn] = (round(_a * (_f_ghz_c5 ** _b), 6), round(_c * (_f_ghz_c5 ** _d), 8), _s, _xpd)

_MAT_REMAP_019 = {
    'itu_wood'        : 'itu_plywood',
    'itu_water'       : 'itu_medium_dry_ground',
    'itu_vegetation'  : 'itu_ceiling_board',
    'itu_asphalt'     : 'itu_very_dry_ground',
    'itu_marble'      : 'itu_concrete',
    'itu_plasterboard': 'itu_ceiling_board',
}
TERRAIN_MATERIAL = 'itu_wet_ground'

_ITU_COLOURS_019 = {
    'itu_concrete'         : '0.539 0.539 0.539',
    'itu_brick'            : '1.000 0.498 0.055',
    'itu_glass'            : '0.596 0.875 0.541',
    'itu_plywood'          : '0.514 0.376 0.220',
    'itu_metal'            : '0.220 0.220 0.254',
    'itu_wet_ground'       : '0.910 0.569 0.055',
    'itu_very_dry_ground'  : '0.498 0.498 0.498',
    'itu_medium_dry_ground': '0.780 0.780 0.780',
    'itu_ceiling_board'    : '0.180 0.450 0.180',
}

used_mats = set(mat_plys.keys()) | {TERRAIN_MATERIAL}
_used_mats_remapped = {_MAT_REMAP_019.get(m, m) for m in used_mats} | {TERRAIN_MATERIAL}

lines = ['<?xml version="1.0" encoding="utf-8"?>', '<scene version="2.1.0">', '',
          '  <!-- ── ITU-R P.2040-2 Materials ────────────────────── -->']
for mat_name in sorted(_used_mats_remapped):
    rgb = _ITU_COLOURS_019.get(mat_name, '0.5 0.5 0.5')
    lines += [f'  <bsdf type="diffuse" id="{mat_name}">',
              f'    <rgb name="reflectance" value="{rgb}"/>',
              '  </bsdf>', '']

lines += ['  <!-- ── Terrain ─────────────────────────────────────── -->',
          '  <shape type="ply" id="mesh-ground">',
          '    <string name="filename" value="meshes/terrain.ply"/>',
          f'    <ref id="{TERRAIN_MATERIAL}" name="bsdf"/>',
          '    <boolean name="face_normals" value="true"/>',
          '  </shape>', '']

lines.append('  <!-- ── USD-imported geometry ───────────────────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    ref_mat = _MAT_REMAP_019.get(mat_name, mat_name)
    for ply_path, role in ply_list:
        mesh_id = 'mesh-' + ply_path.split('/')[-1].replace('.ply', '')
        lines += [f'  <shape type="ply" id="{mesh_id}">',
                  f'    <string name="filename" value="{ply_path}"/>',
                  f'    <ref id="{ref_mat}" name="bsdf"/>',
                  '    <boolean name="face_normals" value="true"/>',
                  '  </shape>']
lines += ['', '</scene>']

scene_xml = os.path.join(SCENE_DIR, 'scene.xml')
with open(scene_xml, 'w') as f:
    f.write('\n'.join(lines))

total_shapes = 1 + sum(len(v) for v in mat_plys.values())
print(f'Wrote: {scene_xml}')
print(f'  Materials : {len(used_mats)}')
print(f'  Shapes    : {total_shapes}  (1 terrain + {total_shapes - 1} USD-imported parts)')

meta = {'utm_epsg': UTM_EPSG, 'projection_crs': PROJECTION_CRS,
        'source': 'usd_omniverse', 'demo_mode': DEMO_MODE,
        'n_meshes': len(usd_meshes)}
with open(os.path.join(BASE_DIR, 'scene_parameters.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Metadata  : {os.path.join(BASE_DIR, "scene_parameters.json")}')


## CELL 6 — 2D Top-Down Preview

In [ ]:
# ============================================================
# CELL 6 — 2D PREVIEW OF EXTRACTED GEOMETRY
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))
_cmap = plt.get_cmap('tab10')
_mat_colors = {m: _cmap(i % 10) for i, m in enumerate(sorted({mesh_itu_material[m['name']] for m in usd_meshes}))}

for m in usd_meshes:
    v = m['verts']
    xs, ys = v[:, 0], v[:, 1]
    hull_order = np.argsort(np.arctan2(ys - ys.mean(), xs - xs.mean()))
    ax.fill(xs[hull_order], ys[hull_order], alpha=0.5,
            color=_mat_colors[mesh_itu_material[m['name']]], edgecolor='k', linewidth=0.5)
    ax.text(xs.mean(), ys.mean(), m['name'], fontsize=7, ha='center')

handles = [mpatches.Patch(color=c, label=m) for m, c in _mat_colors.items()]
ax.legend(handles=handles, loc='upper right', fontsize=8)
ax.set_xlabel('Local X (m)'); ax.set_ylabel('Local Y (m)')
ax.set_title(f'USD scene preview ({"DEMO" if DEMO_MODE else USD_FILE})  --  {len(usd_meshes)} prims')
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
_png = os.path.join(SCENE_DIR, 'usd_scene_preview.png')
plt.savefig(_png, dpi=130)
plt.close()
try:
    from IPython.display import Image, display
    display(Image(filename=_png, width=700))
except Exception:
    pass
print(f'Saved preview -> {_png}')


## CELL 7 — Load into Sionna RT and Sanity-Check
Requires Sionna to be installed; safe to skip if you only want to inspect the
written `scene.xml`/PLYs.

In [ ]:
# ============================================================
# CELL 7 — LOAD INTO SIONNA RT
# ============================================================
try:
    import sionna
    from sionna.rt import load_scene
    print(f'Sionna : {sionna.__version__}')
    scene = load_scene(scene_xml)
    print(f'Loaded : {scene_xml}')
    print(f'Radio materials ({len(scene.radio_materials)}): {sorted(scene.radio_materials.keys())}')
    print(f'Scene objects ({len(scene.objects)}): {sorted(scene.objects.keys())}')
    bbox = scene.mi_scene.bbox()
    print(f'BBox   : min={list(bbox.min)}  max={list(bbox.max)}')
except ImportError as _e:
    print(f'Sionna not installed in this environment -- skipping load check ({_e})')
except Exception as _e:
    print(f'Scene load FAILED: {_e}')
    import traceback; traceback.print_exc()
